## Оценка ансамбля BERT + TF-IDF+LogReg на валидационной выборке
Скрипт загружает валидационный датасет и конфиг ансамбля, по нему поднимает BERT-модель и TF-IDF+LogReg пайплайн, считает для них вероятности классов и сравнивает качество отдельных моделей и их взвешенного ансамбля. Для каждой из трёх схем (чистый BERT, чистый TF-IDF+LR, ансамбль) вычисляются macro-F1 и подробный classification report на `val_super`.

#### Итоговые результаты оценки моделей на валидации
На валидационном наборе данных была проведена оценка трёх вариантов моделей: отдельного BERT-классификатора, отдельного TF-IDF+LogReg пайплайна и взвешенного ансамбля этих моделей. Лучшее качество показывают BERT-эмбеддинги, тогда как ансамбль, несмотря на добавление TF-IDF+LR, не превосходит BERT-модель.

**Сводка macro-F1 на val_super:**
- **BERT:** 0.7338 — наиболее сильная модель, показывающая устойчивое качество по всем трём классам  
- **TF-IDF + Logistic Regression:** 0.6222 — заметно слабее, особенно на негативном классе  
- **Ансамбль (w_bert=0.35, w_lr=0.65):** 0.6642 — улучшает LR, но остаётся хуже чистого BERT

**Вывод:** BERT-модель остаётся оптимальным выбором, а текущие веса ансамбля не дают прироста качества. Для улучшения ансамбля можно варьировать веса, использовать другие классические модели или построить стекер.


In [ ]:
import os
import json
import gc

import numpy as np
import pandas as pd

from joblib import load
from sklearn.metrics import f1_score, classification_report

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification


VAL_PATH = "../data/processed/val_super.csv"  

ENSEMBLE_CONFIG = "../models/ensemble_config_cv_tiny.json"

MAX_LEN_DEFAULT = 192
BATCH_SIZE_DEFAULT = 32


def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")


class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len: int):
        self.texts = list(texts)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


def get_bert_probs_for_texts(texts: pd.Series,
                             model_dir: str,
                             max_len: int,
                             batch_size: int) -> np.ndarray:
    device = get_device()
    print("Устройство для BERT:", device)

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    ds = InferenceDataset(texts, tokenizer, max_len=max_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    all_probs = []
    with torch.no_grad():
        for i, batch in enumerate(dl):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            probs = softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())

            # для прогресса:
            # if (i + 1) % 100 == 0:
            #     print(f"BERT infer: batch {i+1}/{len(dl)}")

    all_probs = np.vstack(all_probs)

    del model, tokenizer
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return all_probs


def get_joblib_probs_for_texts(texts: pd.Series,
                               joblib_path: str) -> np.ndarray:
    if not os.path.exists(joblib_path):
        raise FileNotFoundError(f"Не найден {joblib_path}")
    pipeline = load(joblib_path)

    if hasattr(pipeline, "predict_proba"):
        probs = pipeline.predict_proba(texts.astype(str))
        return probs
    else:
        preds = pipeline.predict(texts.astype(str))
        classes = np.unique(preds)
        num_classes = len(classes)
        oh = np.zeros((len(preds), num_classes), dtype=float)
        for i, c in enumerate(preds):
            oh[i, int(c)] = 1.0
        return oh


def main():
    if not os.path.exists(VAL_PATH):
        raise FileNotFoundError(f"Не найден {VAL_PATH}")

    print(f"Читаем val из {VAL_PATH}...")
    df = pd.read_csv(VAL_PATH)

    for col in ["text", "label"]:
        if col not in df.columns:
            raise ValueError(f"В val_super должна быть колонка '{col}'")

    texts = df["text"].astype(str)
    y_true = df["label"].astype(int).values

    print("Размер val_super:", len(texts))
    print("Распределение классов на val:")
    print(df["label"].value_counts().sort_index())

    if not os.path.exists(ENSEMBLE_CONFIG_PATH):
        raise FileNotFoundError(f"Не найден {ENSEMBLE_CONFIG_PATH}")

    with open(ENSEMBLE_CONFIG_PATH, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    print("\nКонфиг ансамбля:")
    print(json.dumps(cfg, ensure_ascii=False, indent=2))

    # достаём пути и веса
    bert_model_dir = cfg["bert_model_dir"]
    lr_model_path = cfg["lr_model_path"]
    w_bert = cfg["bert_weight"]
    w_lr = cfg["lr_weight"]
    max_len = cfg.get("max_len", MAX_LEN_DEFAULT)
    batch_size = cfg.get("batch_size", BATCH_SIZE_DEFAULT)

    cfg_dir = os.path.dirname(os.path.abspath(ENSEMBLE_CONFIG_PATH))
    if not os.path.isabs(bert_model_dir):
        bert_model_dir = os.path.join(cfg_dir, bert_model_dir)
    if not os.path.isabs(lr_model_path):
        lr_model_path = os.path.join(cfg_dir, lr_model_path)

    print("\nИспользуемая BERT-модель :", bert_model_dir)
    print("Используемый LR-пайплайн:", lr_model_path)
    print(f"Веса ансамбля: w_bert={w_bert:.3f}, w_lr={w_lr:.3f}")
    print(f"max_len={max_len}, batch_size={batch_size}")

    # считаем пробы BERT и LR на val ===

    print("\nСчитаем BERT-пробы на val_super...")
    bert_probs = get_bert_probs_for_texts(
        texts,
        bert_model_dir,
        max_len=max_len,
        batch_size=batch_size,
    )

    print("Считаем TF-IDF+LR-пробы на val_super...")
    lr_probs = get_joblib_probs_for_texts(texts, lr_model_path)

    assert bert_probs.shape == lr_probs.shape, \
        f"bert_probs shape {bert_probs.shape} != lr_probs shape {lr_probs.shape}"

    # === Оценка: чистый BERT ===

    bert_preds = bert_probs.argmax(axis=1)
    bert_f1 = f1_score(y_true, bert_preds, average="macro")

    print("\n===== Оценка ЧИСТОГО BERT на val_super =====")
    print(f"macro-F1: {bert_f1:.4f}")
    print("Classification report (BERT):")
    print(classification_report(y_true, bert_preds, digits=4))

    # === Оценка: чистый LR ===

    lr_preds = lr_probs.argmax(axis=1)
    lr_f1 = f1_score(y_true, lr_preds, average="macro")

    print("\n===== Оценка ЧИСТОГО TF-IDF+LR на val_super =====")
    print(f"macro-F1: {lr_f1:.4f}")
    print("Classification report (LR):")
    print(classification_report(y_true, lr_preds, digits=4))

    # === Оценка: АНСАМБЛЬ (BERT + LR) ===

    ensemble_probs = w_bert * bert_probs + w_lr * lr_probs
    ensemble_preds = ensemble_probs.argmax(axis=1)
    ensemble_f1 = f1_score(y_true, ensemble_preds, average="macro")

    print("\n===== Оценка АНСАМБЛЯ на val_super =====")
    print(f"macro-F1: {ensemble_f1:.4f}")
    print("Classification report (ENSEMBLE):")
    print(classification_report(y_true, ensemble_preds, digits=4))

    print("\nСводка по macro-F1 на val_super:")
    print(f"  BERT      : {bert_f1:.4f}")
    print(f"  TF-IDF+LR : {lr_f1:.4f}")
    print(f"  ENSEMBLE  : {ensemble_f1:.4f}")


if __name__ == "__main__":
    main()


Читаем val из ../data/processed/val_super.csv...
Размер val_super: 11771
Распределение классов на val:
label
0    3896
1    3937
2    3938
Name: count, dtype: int64

Конфиг ансамбля:
{
  "bert_model_dir": "../models/rurorberta_base2_manual",
  "lr_model_path": "../models/sentiment_lr_optuna.joblib",
  "bert_weight": 0.35000000000000003,
  "lr_weight": 0.6499999999999999,
  "num_labels": 3,
  "max_len": 192,
  "batch_size": 32,
  "n_splits": 5,
  "cv_f1_macro": 0.90743968911667
}

Используемая BERT-модель : /Users/olgashalashova/sentiment_project/models/../models/rurorberta_base2_manual
Используемый LR-пайплайн: /Users/olgashalashova/sentiment_project/models/../models/sentiment_lr_optuna.joblib
Веса ансамбля: w_bert=0.350, w_lr=0.650
max_len=192, batch_size=32

Считаем BERT-пробы на val_super...
Устройство для BERT: mps


The tokenizer you are loading from '/Users/olgashalashova/sentiment_project/models/../models/rurorberta_base2_manual' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Считаем TF-IDF+LR-пробы на val_super...

===== Оценка ЧИСТОГО BERT на val_super =====
macro-F1: 0.7338
Classification report (BERT):
              precision    recall  f1-score   support

           0     0.6357    0.6360    0.6359      3896
           1     0.8186    0.8174    0.8180      3937
           2     0.7471    0.7478    0.7475      3938

    accuracy                         0.7341     11771
   macro avg     0.7338    0.7338    0.7338     11771
weighted avg     0.7341    0.7341    0.7341     11771


===== Оценка ЧИСТОГО TF-IDF+LR на val_super =====
macro-F1: 0.6222
Classification report (LR):
              precision    recall  f1-score   support

           0     0.5125    0.5331    0.5226      3896
           1     0.7076    0.7148    0.7111      3937
           2     0.6496    0.6171    0.6329      3938

    accuracy                         0.6220     11771
   macro avg     0.6232    0.6216    0.6222     11771
weighted avg     0.6236    0.6220    0.6226     11771


===== Оц